In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

keras.utils.set_random_seed(42)
np.random.seed(42)

In [ ]:
(raw_train_images, train_labels), (raw_test_images, test_labels) = keras.datasets.mnist.load_data()
print(raw_train_images.shape, raw_test_images.shape)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for axis, index in zip(axes.flatten(), range(20)):
    axis.imshow(raw_train_images[index], cmap="gray")
    axis.set_title(train_labels[index])
    axis.axis("off")
plt.show()

plt.figure(figsize=(6, 3))
pd.Series(train_labels).value_counts().sort_index().plot(kind="bar", rot=0)
plt.title("Digits in training set")
plt.show()

In [ ]:
raw_train_flat = raw_train_images.reshape(len(raw_train_images), -1).astype("float32")
raw_test_flat = raw_test_images.reshape(len(raw_test_images), -1).astype("float32")

scaled_train = raw_train_flat / 255.0
scaled_test = raw_test_flat / 255.0

pixel_mean = scaled_train.mean()
pixel_std = scaled_train.std()
standardized_train = (scaled_train - pixel_mean) / pixel_std
standardized_test = (scaled_test - pixel_mean) / pixel_std

for name, values in [("raw 0-255", raw_train_flat), ("scaled 0-1", scaled_train), ("standardized", standardized_train)]:
    print(f"{name:<13} min {values.min():8.3f}  max {values.max():8.3f}  mean {values.mean():8.3f}  std {values.std():8.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for axis, (name, values) in zip(axes, [("raw 0-255", raw_train_flat), ("scaled 0-1", scaled_train), ("standardized", standardized_train)]):
    axis.hist(values[:2000].flatten(), bins=50, color="gray")
    axis.set_title(name)
    axis.set_yscale("log")
plt.show()

In [ ]:
def softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)

def cross_entropy(probabilities, labels):
    return -np.mean(np.log(probabilities[np.arange(len(labels)), labels] + 1e-12))

example_logits = np.array([[2.0, 1.0, 0.1], [1000.0, 999.0, 998.0]])
print("softmax:", softmax(example_logits).round(4))
print("each row sums to:", softmax(example_logits).sum(axis=1))

confident_right = np.array([[0.05, 0.9, 0.05]])
unsure = np.array([[0.3, 0.4, 0.3]])
confident_wrong = np.array([[0.9, 0.05, 0.05]])
for name, probabilities in [("confident and right", confident_right), ("unsure", unsure), ("confident and wrong", confident_wrong)]:
    print(f"{name:<20} cross entropy {cross_entropy(probabilities, np.array([1])):.4f}")

In [ ]:
class FeedforwardNetwork:
    def __init__(self, input_size=784, hidden_size=128, output_size=10, seed=0):
        generator = np.random.default_rng(seed)
        self.w1 = generator.normal(0, np.sqrt(2 / input_size), (input_size, hidden_size))
        self.b1 = np.zeros(hidden_size)
        self.w2 = generator.normal(0, np.sqrt(2 / hidden_size), (hidden_size, output_size))
        self.b2 = np.zeros(output_size)

    def forward(self, images):
        hidden_input = images @ self.w1 + self.b1
        hidden_output = np.maximum(0, hidden_input)
        logits = hidden_output @ self.w2 + self.b2
        probabilities = softmax(logits)
        return probabilities, (images, hidden_input, hidden_output, probabilities)

    def backward(self, labels, cache):
        images, hidden_input, hidden_output, probabilities = cache
        batch = len(labels)
        logit_gradient = probabilities.copy()
        logit_gradient[np.arange(batch), labels] -= 1
        logit_gradient /= batch
        hidden_gradient = logit_gradient @ self.w2.T
        hidden_gradient[hidden_input <= 0] = 0
        return {
            "w2": hidden_output.T @ logit_gradient,
            "b2": logit_gradient.sum(axis=0),
            "w1": images.T @ hidden_gradient,
            "b1": hidden_gradient.sum(axis=0)
        }

    def update(self, gradients, learning_rate):
        self.w1 -= learning_rate * gradients["w1"]
        self.b1 -= learning_rate * gradients["b1"]
        self.w2 -= learning_rate * gradients["w2"]
        self.b2 -= learning_rate * gradients["b2"]

    def predict(self, images):
        return self.forward(images)[0].argmax(axis=1)

In [ ]:
check_network = FeedforwardNetwork(seed=1)
check_images = scaled_train[:64].astype(np.float64)
check_labels = train_labels[:64]
_, check_cache = check_network.forward(check_images)
analytic_gradients = check_network.backward(check_labels, check_cache)

epsilon = 1e-5
for _ in range(6):
    row = np.random.randint(check_network.w2.shape[0])
    column = np.random.randint(check_network.w2.shape[1])
    original_value = check_network.w2[row, column]
    check_network.w2[row, column] = original_value + epsilon
    loss_plus = cross_entropy(check_network.forward(check_images)[0], check_labels)
    check_network.w2[row, column] = original_value - epsilon
    loss_minus = cross_entropy(check_network.forward(check_images)[0], check_labels)
    check_network.w2[row, column] = original_value
    numeric_gradient = (loss_plus - loss_minus) / (2 * epsilon)
    print(f"w2[{row:>3},{column}]  analytic {analytic_gradients['w2'][row, column]: .8f}  numeric {numeric_gradient: .8f}")

In [ ]:
numpy_network = FeedforwardNetwork(seed=42)
learning_rate = 0.1
batch_size = 128
epochs = 10
numpy_history = {"train_loss": [], "test_accuracy": []}

for epoch in range(epochs):
    order = np.random.permutation(len(scaled_train))
    start_time = time.time()
    batch_losses = []
    for start in range(0, len(order), batch_size):
        batch_indices = order[start:start + batch_size]
        probabilities, cache = numpy_network.forward(scaled_train[batch_indices])
        batch_losses.append(cross_entropy(probabilities, train_labels[batch_indices]))
        numpy_network.update(numpy_network.backward(train_labels[batch_indices], cache), learning_rate)
    test_accuracy = np.mean(numpy_network.predict(scaled_test) == test_labels)
    numpy_history["train_loss"].append(np.mean(batch_losses))
    numpy_history["test_accuracy"].append(test_accuracy)
    print(f"epoch {epoch + 1:>2}  loss {numpy_history['train_loss'][-1]:.4f}  test accuracy {test_accuracy:.4f}  {time.time() - start_time:.1f}s")

In [ ]:
def build_feedforward(hidden_units=(256, 128), learning_rate=1e-3, optimizer_name="adam"):
    model = keras.Sequential([layers.Input(shape=(784,))])
    for units in hidden_units:
        model.add(layers.Dense(units, activation="relu"))
    model.add(layers.Dense(10, activation="softmax"))
    optimizer = keras.optimizers.Adam(learning_rate) if optimizer_name == "adam" else keras.optimizers.SGD(learning_rate)
    model.compile(optimizer=optimizer, loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    return model

normalization_histories = {}
for name, train_inputs, test_inputs in [("raw 0-255", raw_train_flat, raw_test_flat), ("scaled 0-1", scaled_train, scaled_test), ("standardized", standardized_train, standardized_test)]:
    keras.utils.set_random_seed(42)
    model = build_feedforward(learning_rate=0.01, optimizer_name="sgd")
    history = model.fit(train_inputs, train_labels, validation_data=(test_inputs, test_labels), epochs=8, batch_size=128, verbose=0)
    normalization_histories[name] = history.history
    print(f"{name:<13} final train loss {history.history['loss'][-1]:.4f}  test accuracy {history.history['val_accuracy'][-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for name, history in normalization_histories.items():
    axes[0].plot(history["loss"], marker="o", label=name)
    axes[1].plot(history["val_accuracy"], marker="o", label=name)
axes[0].set_title("Training loss with SGD (lr 0.01)")
axes[0].set_yscale("log")
axes[1].set_title("Test accuracy")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.legend()
plt.show()

In [ ]:
keras.utils.set_random_seed(42)
digit_model = build_feedforward()
digit_model.summary()
digit_history = digit_model.fit(
    scaled_train, train_labels,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
)
test_loss, test_accuracy = digit_model.evaluate(scaled_test, test_labels, verbose=0)
print(f"keras feedforward test accuracy {test_accuracy:.4f}  test loss {test_loss:.4f}")
print(f"numpy feedforward test accuracy {numpy_history['test_accuracy'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(digit_history.history["loss"], label="keras train")
axes[0].plot(digit_history.history["val_loss"], label="keras validation")
axes[0].plot(numpy_history["train_loss"], label="numpy train", linestyle="--")
axes[0].set_title("Cross-entropy loss")
axes[0].legend()
axes[1].plot(digit_history.history["accuracy"], label="keras train")
axes[1].plot(digit_history.history["val_accuracy"], label="keras validation")
axes[1].plot(numpy_history["test_accuracy"], label="numpy test", linestyle="--")
axes[1].set_title("Accuracy")
axes[1].legend()
plt.show()

test_probabilities = digit_model.predict(scaled_test, verbose=0)
manual_loss = cross_entropy(test_probabilities.astype(np.float64), test_labels)
keras_loss = float(keras.losses.SparseCategoricalCrossentropy()(test_labels, test_probabilities))
print(f"manual cross entropy {manual_loss:.5f}  keras cross entropy {keras_loss:.5f}")
print("softmax outputs sum to one:", np.allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-4))

In [ ]:
test_predictions = test_probabilities.argmax(axis=1)
print(classification_report(test_labels, test_predictions, digits=4))

plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(test_labels, test_predictions), annot=True, fmt="d", cmap="Blues")
plt.xlabel("predicted")
plt.ylabel("actual")
plt.title("MNIST confusion matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for column, index in enumerate(np.random.choice(len(raw_test_images), 4, replace=False)):
    axes[0, column].imshow(raw_test_images[index], cmap="gray")
    axes[0, column].set_title(f"actual {test_labels[index]}  predicted {test_predictions[index]}")
    axes[0, column].axis("off")
    axes[1, column].bar(range(10), test_probabilities[index], color="steelblue")
    axes[1, column].set_xticks(range(10))
    axes[1, column].set_ylim(0, 1)
    axes[1, column].set_title("softmax probabilities")
plt.tight_layout()
plt.show()

wrong_indices = np.where(test_predictions != test_labels)[0]
print("misclassified test digits:", len(wrong_indices))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for axis, index in zip(axes.flatten(), wrong_indices[:16]):
    axis.imshow(raw_test_images[index], cmap="gray")
    axis.set_title(f"{test_labels[index]} -> {test_predictions[index]} ({test_probabilities[index].max():.2f})", color="red", fontsize=9)
    axis.axis("off")
plt.tight_layout()
plt.show()

digit_model.save("mnist_feedforward.keras")